In [6]:
import certifi 
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langsmith import Client
from langchain.tools import tool
import requests

In [7]:
from langchain.agents import create_agent

In [18]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY=os.getenv("WEATHERSTACK_API_KEY")
LANGSMITH_API_KEY=os.getenv("LANGSMITH_API_KEY")


In [17]:
search_tool = TavilySearchResults(max_results=5)

In [13]:
result = search_tool.invoke("Give me the latest news on AI")
result

[{'title': 'Artificial intelligence',
  'url': 'https://en.wikipedia.org/wiki/Artificial_intelligence',
  'content': 'In November 2023, the first global AI Safety Summit was held in Bletchley Park in the UK to discuss the near and far term risks of AI and the possibility of mandatory and voluntary regulatory frameworks.( 28 countries including the United States, China, and the European Union issued a declaration at the start of the summit, calling for international co-operation to manage the challenges and risks of artificial intelligence.( In May 2024 at the AI Seoul Summit, 16 global AI tech companies agreed to safety commitments on the development of AI.( [...] ### Web search\n\nMicrosoft introduced Copilot Search in February 2023 under the name Bing Chat. Copilot Search provides AI-generated summaries.(\n\nGenerative AI features added to Google Search include AI Overviews in May 2024( and AI Mode in May 2025.(\n\n### Sexuality\n\nApplications of AI in this domain include AI-enabled

In [22]:
@tool
def get_weather_data(city  : str):   
    """
    Fetch current weather information for a city.
    """
    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data=response.json()

    if "current" not in data :
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [11]:
llm = ChatOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
    model = "openai/gpt-oss-120b"
)


In [5]:
response = llm.invoke("Tell me a joke about AI")
response

AIMessage(content='Why did the AI go to therapy?  \n\nBecause it kept over‑thinking its “deep” feelings and couldn’t stop “re‑training” its emotions! 😄', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 77, 'total_tokens': 147, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 27, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': None, 'queue_time': 0.047929637, 'prompt_time': 0.003313074, 'completion_time': 0.144226125, 'total_time': 0.147539199}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8ea50c0161', 'id': 'chatcmpl-5bd8a685-21d2-464d-a611-bf29633a4f2b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'normalized_finish_reason': 'stop'}, id='lc_run--01a1fc8c-0edc-73a4-9609-675339788c1c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 

In [19]:
# ==========================================
# PROMPT
# ==========================================

client = Client(api_key=os.getenv("LANGSMITH_API_KEY"))
prompt =client.pull_prompt("agentic_test")

In [27]:
print(type(prompt))
print(prompt.input_variables)
print(prompt.messages)

<class 'langchain_core.prompts.chat.ChatPromptTemplate'>
['question']
[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a chatbot.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})]


In [20]:
print(prompt)

input_variables=['question'] input_types={} partial_variables={} metadata={'lc_hub_owner': '-', 'lc_hub_repo': 'agentic_test', 'lc_hub_commit_hash': '970e1fde3d7d79db21acae38e9494c29359ba1d434730e8855320e4bd325445d'} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a chatbot.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})]


In [23]:
agent= create_agent(
    tools=[get_weather_data],
    model=llm,
)

In [26]:
# ==========================================
# RUN
# ==========================================
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Find the capital of egypt and then find its current weather."
        }
    ]
})
print(response["messages"][-1].content)

The capital of Egypt is **Cairo**.  

**Current weather in Cairo (as of the latest data):**  
- **Temperature:** 20 °C  
- **Conditions:** Partly cloudy  
- **Humidity:** 54%
